# Produccion del dataset ∂PV — sesion s3

Cuaderno autonomo para Kaggle: **Accelerator = None**, **Internet = On**, y luego *Save Version → Save & Run All (Commit)*.

Bloques de esta sesion (1024 dispositivos Sobol cada uno, sin solaparse con ninguna otra sesion): `E2A-b000`, `E2A-b001`, `E2A-b002`, `E2B-b000`, `E2B-b001`, `E2B-b002`. El refinado 4x va solo en los bloques b000.

Salida en `/kaggle/working`: un `dpv_<familia><etapa>_b<nnn>.h5` por bloque y `resumen_sesion.json`, que dice que bloques quedaron completos. Si se acaba el presupuesto de 11 h, los bloques que falten quedan marcados para otra sesion.

In [ ]:
# Codigo del generador fijado al commit verificado por los smokes
import subprocess, sys, pathlib
COMMIT = "fbc352679f9d644cb781fd98aec4401e7cf73af2"
ROOT = pathlib.Path("/tmp/dpv-dataset")
if not ROOT.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/smindiol/dpv-dataset", str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", COMMIT], check=True)
sys.path.insert(0, str(ROOT))
print(subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
import os, json

MODO = "PRODUCCION"
UNIDADES = ["E2A-b000", "E2A-b001", "E2A-b002", "E2B-b000", "E2B-b001", "E2B-b002"]
PLATAFORMA = "cpu"                 # bandera: "cpu" o "gpu"
N_TRABAJADORES = None              # None = un proceso por nucleo
PRESUPUESTO_HORAS = 11.0           # Kaggle corta la sesion a las 12 h
REFINAR_SOLO_BLOQUE0 = True        # el refinado 4x solo en b000 (cuesta ~20x)
DIR_SALIDA = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
os.environ["FORGE_CLONE_COMMIT"] = COMMIT   # queda en resumen_sesion.json
print(f"MODO={MODO} UNIDADES={UNIDADES} PLATAFORMA={PLATAFORMA} salida en {DIR_SALIDA}")

In [ ]:
# deltapv fijado al commit verificado; --no-deps para no reemplazar el jax con CUDA de la imagen.
!pip install -q --no-deps "git+https://github.com/romanodev/deltapv@f874e24"
import h5py, scipy, numpy
print("h5py", h5py.__version__, "| scipy", scipy.__version__, "| numpy", numpy.__version__)

In [ ]:
from dpvgen import orquestador

orquestador.configurar(PLATAFORMA=PLATAFORMA, N_TRABAJADORES=N_TRABAJADORES,
                       PRESUPUESTO_HORAS=PRESUPUESTO_HORAS, DIR_SALIDA=DIR_SALIDA,
                       REFINAR_SOLO_BLOQUE0=REFINAR_SOLO_BLOQUE0)
if MODO == "TEST":
    RESUMEN = orquestador.ejecutar_test_rapido(PLATAFORMA, 20)
elif MODO == "PRODUCCION":
    RESUMEN = orquestador.ejecutar_unidades(UNIDADES)
else:
    raise ValueError(MODO)


In [ ]:
# Verificacion de lo producido: tasa de cada estado (r04 exige reportarla) y PCE.
import glob
import numpy as np
from collections import Counter

for ruta in sorted(glob.glob(os.path.join(DIR_SALIDA, "*.h5"))):
    with h5py.File(ruta, "r") as f:
        print(f"\n{os.path.basename(ruta)}  familia={f.attrs['familia']} etapa={f.attrs['etapa']}  "
              f"{os.path.getsize(ruta)/1e9:.2f} GB")
        print("  xi:", [p["nombre"] for p in json.loads(f.attrs["parametros_xi"])])
        for g in ("sims", "esquinas", "refinados"):
            est = Counter(f[g][k].attrs["estado"] for k in f[g])
            n = sum(est.values())
            print(f"  /{g}: {n}  " + ", ".join(f"{e}={c} ({100*c/max(n,1):.1f}%)" for e, c in est.items()))
        oks = [k for k in f["sims"] if f["sims"][k].attrs["estado"] == "ok"]
        if oks:
            eff = np.array([f["sims"][k].attrs["eff"] for k in oks]) * 100
            nv = np.array([f["sims"][k].attrs["n_voltajes"] for k in oks])
            print(f"  PCE %: p5={np.percentile(eff,5):.2f} p50={np.median(eff):.2f} "
                  f"p95={np.percentile(eff,95):.2f} | voltajes por curva: {nv.min()}-{nv.max()}")